# CUDA Programming Study Manual
## 4. List Ranking, 5. Sparse Matrix Computation, 6. Convolution Kernel

**Target:** Google Colab / Linux system with NVIDIA CUDA Toolkit and `nvcc`.

This notebook demonstrates three important parallel algorithms using **CUDA C/C++ kernels**, compiled and executed from a Python/Jupyter notebook.

### Learning outcomes
1. Understand the idea of **list ranking** and parallel pointer jumping.
2. Implement **Sparse Matrix–Vector Multiplication (SpMV)** using CSR format.
3. Implement a **2-D convolution kernel** in CUDA.
4. Compare CUDA results with CPU reference implementations.
5. Understand grid/block/thread organization and basic CUDA memory management.

In [ ]:
# Check Python environment
import numpy as np
import os, subprocess, textwrap, json, math

print("NumPy:", np.__version__)
print(subprocess.run(["bash", "-lc", "nvcc --version"], capture_output=True, text=True).stdout)

## 4. List Ranking

### Problem

A linked list contains nodes where each node points to its successor:

`0 -> 1 -> 2 -> 3 -> 4 -> -1`

The **rank** of a node is its distance from the end of the list:

- node 4 → rank 0
- node 3 → rank 1
- node 2 → rank 2
- node 1 → rank 3
- node 0 → rank 4

A sequential traversal is inherently serial. CUDA can accelerate the computation using **pointer jumping**.

### Pointer jumping idea

For each node `i`:

```text
rank[i] = 1
next[i] = successor[i]

repeat:
    next[i] = next[next[i]]
    rank[i] += rank[next[i]]
```

A convenient implementation uses two arrays (`next` and `new_next`) so that all threads operate on the same logical iteration.

In [ ]:
%%writefile list_ranking.cu
#include <cuda_runtime.h>
#include <iostream>
#include <vector>
#include <cstdlib>

#define CUDA_CHECK(call) do {                                      \
    cudaError_t err = call;                                        \
    if (err != cudaSuccess) {                                      \
        std::cerr << "CUDA error: " << cudaGetErrorString(err)      \
                  << " at line " << __LINE__ << std::endl;         \
        exit(EXIT_FAILURE);                                        \
    }                                                               \
} while(0)

// Pointer jumping: each active node jumps over its successor.
__global__ void listRankKernel(const int* next, int* newNext,
                               const int* rank, int* newRank,
                               int n) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;

    if (i < n) {
        int s = next[i];

        if (s != -1) {
            int ss = next[s];

            // If a successor exists, accumulate its rank.
            newRank[i] = rank[i] + rank[s];
            newNext[i] = ss;
        } else {
            newRank[i] = rank[i];
            newNext[i] = -1;
        }
    }
}

int main() {
    // Example list:
    // 0 -> 1 -> 2 -> 3 -> 4 -> 5 -> -1
    const int n = 6;
    std::vector<int> h_next = {1, 2, 3, 4, 5, -1};
    std::vector<int> h_rank(n, 0);

    // Initial rank contribution:
    // every non-tail node contributes 1 edge.
    for (int i = 0; i < n; ++i)
        h_rank[i] = (h_next[i] == -1) ? 0 : 1;

    int *d_next, *d_newNext, *d_rank, *d_newRank;

    CUDA_CHECK(cudaMalloc(&d_next, n * sizeof(int)));
    CUDA_CHECK(cudaMalloc(&d_newNext, n * sizeof(int)));
    CUDA_CHECK(cudaMalloc(&d_rank, n * sizeof(int)));
    CUDA_CHECK(cudaMalloc(&d_newRank, n * sizeof(int)));

    CUDA_CHECK(cudaMemcpy(d_next, h_next.data(),
                          n * sizeof(int), cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_rank, h_rank.data(),
                          n * sizeof(int), cudaMemcpyHostToDevice));

    int blockSize = 256;
    int gridSize = (n + blockSize - 1) / blockSize;

    // Pointer jumping converges in O(log n) iterations.
    int iterations = 0;
    int active = n;

    while (active > 1) {
        listRankKernel<<<gridSize, blockSize>>>(
            d_next, d_newNext, d_rank, d_newRank, n
        );

        CUDA_CHECK(cudaGetLastError());
        CUDA_CHECK(cudaDeviceSynchronize());

        std::swap(d_next, d_newNext);
        std::swap(d_rank, d_newRank);

        iterations++;

        // For demonstration we use ceil(log2(n)) iterations.
        if (iterations >= 10) break;
        active = (active + 1) / 2;
    }

    CUDA_CHECK(cudaMemcpy(h_rank.data(), d_rank,
                          n * sizeof(int), cudaMemcpyDeviceToHost));

    std::cout << "List Ranking Result\n";
    for (int i = 0; i < n; ++i)
        std::cout << "Node " << i << " -> rank " << h_rank[i] << "\n";

    CUDA_CHECK(cudaFree(d_next));
    CUDA_CHECK(cudaFree(d_newNext));
    CUDA_CHECK(cudaFree(d_rank));
    CUDA_CHECK(cudaFree(d_newRank));

    return 0;
}

In [ ]:
!nvcc -O2 list_ranking.cu -o list_ranking
!./list_ranking

### Expected result

For:

`0 -> 1 -> 2 -> 3 -> 4 -> 5 -> -1`

the expected ranks are:

```text
Node 0 -> 5
Node 1 -> 4
Node 2 -> 3
Node 3 -> 2
Node 4 -> 1
Node 5 -> 0
```

> **Important:** In production-quality list-ranking implementations, the termination condition is normally based on the list structure or a fixed `ceil(log2(n))` pointer-jumping schedule. The example keeps the kernel simple for teaching.

## 5. Sparse Matrix Computation

A sparse matrix contains mostly zero values. Storing all zero elements wastes memory.

For example:

```text
[10  0  0  2]
[ 0  3  0  0]
[ 0  0  5  0]
[ 4  0  0  6]
```

can be represented using **CSR (Compressed Sparse Row)**:

- `values` = non-zero values
- `colIndex` = column of each non-zero value
- `rowPtr` = start/end position of each row

### CSR SpMV

We compute:

`y = A × x`

Each CUDA thread processes one matrix row:

```text
for each non-zero element in row:
    y[row] += value[k] * x[colIndex[k]]
```

This is called **Sparse Matrix–Vector Multiplication (SpMV)**.

In [ ]:
%%writefile sparse_spmv.cu
#include <cuda_runtime.h>
#include <iostream>
#include <vector>
#include <cmath>
#include <cstdlib>

#define CUDA_CHECK(call) do {                                      \
    cudaError_t err = call;                                        \
    if (err != cudaSuccess) {                                      \
        std::cerr << "CUDA error: " << cudaGetErrorString(err)      \
                  << " at line " << __LINE__ << std::endl;         \
        exit(EXIT_FAILURE);                                        \
    }                                                               \
} while(0)

// One thread processes one row of the sparse matrix.
__global__ void spmvCSRKernel(int rows,
                              const int* rowPtr,
                              const int* colIndex,
                              const float* values,
                              const float* x,
                              float* y) {
    int row = blockIdx.x * blockDim.x + threadIdx.x;

    if (row < rows) {
        float sum = 0.0f;

        for (int k = rowPtr[row]; k < rowPtr[row + 1]; ++k) {
            sum += values[k] * x[colIndex[k]];
        }

        y[row] = sum;
    }
}

int main() {
    // Matrix:
    // [10  0  0  2]
    // [ 0  3  0  0]
    // [ 0  0  5  0]
    // [ 4  0  0  6]

    const int rows = 4;
    const int cols = 4;
    const int nnz = 6;

    std::vector<int> rowPtr   = {0, 2, 3, 4, 6};
    std::vector<int> colIndex = {0, 3, 1, 2, 0, 3};
    std::vector<float> values = {10, 2, 3, 5, 4, 6};

    std::vector<float> x = {1, 2, 3, 4};
    std::vector<float> y(rows, 0);

    int *d_rowPtr, *d_colIndex;
    float *d_values, *d_x, *d_y;

    CUDA_CHECK(cudaMalloc(&d_rowPtr, (rows + 1) * sizeof(int)));
    CUDA_CHECK(cudaMalloc(&d_colIndex, nnz * sizeof(int)));
    CUDA_CHECK(cudaMalloc(&d_values, nnz * sizeof(float)));
    CUDA_CHECK(cudaMalloc(&d_x, cols * sizeof(float)));
    CUDA_CHECK(cudaMalloc(&d_y, rows * sizeof(float)));

    CUDA_CHECK(cudaMemcpy(d_rowPtr, rowPtr.data(),
                          (rows + 1) * sizeof(int),
                          cudaMemcpyHostToDevice));

    CUDA_CHECK(cudaMemcpy(d_colIndex, colIndex.data(),
                          nnz * sizeof(int),
                          cudaMemcpyHostToDevice));

    CUDA_CHECK(cudaMemcpy(d_values, values.data(),
                          nnz * sizeof(float),
                          cudaMemcpyHostToDevice));

    CUDA_CHECK(cudaMemcpy(d_x, x.data(),
                          cols * sizeof(float),
                          cudaMemcpyHostToDevice));

    int blockSize = 256;
    int gridSize = (rows + blockSize - 1) / blockSize;

    spmvCSRKernel<<<gridSize, blockSize>>>(
        rows, d_rowPtr, d_colIndex, d_values, d_x, d_y
    );

    CUDA_CHECK(cudaGetLastError());
    CUDA_CHECK(cudaDeviceSynchronize());

    CUDA_CHECK(cudaMemcpy(y.data(), d_y,
                          rows * sizeof(float),
                          cudaMemcpyDeviceToHost));

    std::cout << "CSR Sparse Matrix-Vector Multiplication\n";
    std::cout << "y = A*x\n";

    for (int i = 0; i < rows; ++i)
        std::cout << "y[" << i << "] = " << y[i] << "\n";

    CUDA_CHECK(cudaFree(d_rowPtr));
    CUDA_CHECK(cudaFree(d_colIndex));
    CUDA_CHECK(cudaFree(d_values));
    CUDA_CHECK(cudaFree(d_x));
    CUDA_CHECK(cudaFree(d_y));

    return 0;
}

In [ ]:
!nvcc -O2 sparse_spmv.cu -o sparse_spmv
!./sparse_spmv

### Manual verification

For:

```text
A =
[10  0  0  2]
[ 0  3  0  0]
[ 0  0  5  0]
[ 4  0  0  6]

x = [1, 2, 3, 4]
```

we get:

```text
y[0] = 10(1) + 2(4) = 18
y[1] =  3(2)       =  6
y[2] =  5(3)       = 15
y[3] =  4(1) + 6(4) = 28
```

Therefore:

`y = [18, 6, 15, 28]`

### Why CSR is useful

For a matrix with millions of elements but only a small percentage of non-zero values:

- memory usage is greatly reduced,
- zero-value multiplications are avoided,
- CUDA threads can process different rows concurrently.

## 6. Convolution Kernel in CUDA

2-D convolution is widely used in:

- image processing,
- edge detection,
- computer vision,
- CNNs,
- filtering and feature extraction.

For an image `I` and kernel `K`:

`O(row,col) = Σ I(row+i,col+j) × K(i,j)`

Each output pixel can be calculated independently, making convolution highly suitable for GPU parallelism.

### Example

A common edge-detection kernel is:

```text
[-1 -1 -1]
[-1  8 -1]
[-1 -1 -1]
```

Each CUDA thread calculates one output pixel.

In [ ]:
%%writefile convolution.cu
#include <cuda_runtime.h>
#include <iostream>
#include <vector>
#include <iomanip>
#include <cstdlib>
#include <cmath>

#define CUDA_CHECK(call) do {                                      \
    cudaError_t err = call;                                        \
    if (err != cudaSuccess) {                                      \
        std::cerr << "CUDA error: " << cudaGetErrorString(err)      \
                  << " at line " << __LINE__ << std::endl;         \
        exit(EXIT_FAILURE);                                        \
    }                                                               \
} while(0)

#define K 3
#define R 1

// One CUDA thread calculates one output pixel.
// Zero padding is used at the image boundary.
__global__ void convolution2DKernel(const float* image,
                                    float* output,
                                    const float* kernel,
                                    int width,
                                    int height) {

    int col = blockIdx.x * blockDim.x + threadIdx.x;
    int row = blockIdx.y * blockDim.y + threadIdx.y;

    if (row < height && col < width) {

        float sum = 0.0f;

        for (int ky = -R; ky <= R; ++ky) {
            for (int kx = -R; kx <= R; ++kx) {

                int inputRow = row + ky;
                int inputCol = col + kx;

                float pixel = 0.0f;

                // Zero padding outside image boundaries.
                if (inputRow >= 0 && inputRow < height &&
                    inputCol >= 0 && inputCol < width) {

                    pixel = image[inputRow * width + inputCol];
                }

                float kval = kernel[(ky + R) * K + (kx + R)];

                sum += pixel * kval;
            }
        }

        output[row * width + col] = sum;
    }
}

int main() {
    const int width = 8;
    const int height = 8;

    // Simple synthetic image.
    std::vector<float> h_image(width * height, 0.0f);

    // Create a bright 4x4 square in the center.
    for (int r = 2; r < 6; ++r) {
        for (int c = 2; c < 6; ++c) {
            h_image[r * width + c] = 10.0f;
        }
    }

    // 3x3 edge detection kernel.
    std::vector<float> h_kernel = {
        -1, -1, -1,
        -1,  8, -1,
        -1, -1, -1
    };

    std::vector<float> h_output(width * height, 0.0f);

    float *d_image, *d_output, *d_kernel;

    CUDA_CHECK(cudaMalloc(&d_image, width * height * sizeof(float)));
    CUDA_CHECK(cudaMalloc(&d_output, width * height * sizeof(float)));
    CUDA_CHECK(cudaMalloc(&d_kernel, K * K * sizeof(float)));

    CUDA_CHECK(cudaMemcpy(d_image, h_image.data(),
                          width * height * sizeof(float),
                          cudaMemcpyHostToDevice));

    CUDA_CHECK(cudaMemcpy(d_kernel, h_kernel.data(),
                          K * K * sizeof(float),
                          cudaMemcpyHostToDevice));

    dim3 block(16, 16);
    dim3 grid((width + block.x - 1) / block.x,
              (height + block.y - 1) / block.y);

    convolution2DKernel<<<grid, block>>>(
        d_image, d_output, d_kernel, width, height
    );

    CUDA_CHECK(cudaGetLastError());
    CUDA_CHECK(cudaDeviceSynchronize());

    CUDA_CHECK(cudaMemcpy(h_output.data(), d_output,
                          width * height * sizeof(float),
                          cudaMemcpyDeviceToHost));

    std::cout << "Convolution Output\n\n";

    for (int r = 0; r < height; ++r) {
        for (int c = 0; c < width; ++c) {
            std::cout << std::setw(6)
                      << std::fixed << std::setprecision(1)
                      << h_output[r * width + c];
        }
        std::cout << "\n";
    }

    CUDA_CHECK(cudaFree(d_image));
    CUDA_CHECK(cudaFree(d_output));
    CUDA_CHECK(cudaFree(d_kernel));

    return 0;
}

In [ ]:
!nvcc -O2 convolution.cu -o convolution
!./convolution

## Python CPU Reference Verification

The following cell independently computes the convolution using NumPy-style Python loops. This is useful for demonstrating that the CUDA kernel produces the expected mathematical result.

In [ ]:
import numpy as np

def convolution_cpu(image, kernel):
    h, w = image.shape
    kh, kw = kernel.shape
    rh, rw = kh // 2, kw // 2

    output = np.zeros_like(image, dtype=np.float32)

    for r in range(h):
        for c in range(w):
            total = 0.0
            for kr in range(kh):
                for kc in range(kw):
                    rr = r + kr - rh
                    cc = c + kc - rw

                    if 0 <= rr < h and 0 <= cc < w:
                        total += image[rr, cc] * kernel[kr, kc]

            output[r, c] = total

    return output

image = np.zeros((8, 8), dtype=np.float32)
image[2:6, 2:6] = 10

kernel = np.array([
    [-1, -1, -1],
    [-1,  8, -1],
    [-1, -1, -1]
], dtype=np.float32)

cpu_output = convolution_cpu(image, kernel)

print(cpu_output)

# CUDA Concepts Used

## 1. Thread indexing

For 1-D problems:

```cpp
int i = blockIdx.x * blockDim.x + threadIdx.x;
```

For 2-D problems:

```cpp
int col = blockIdx.x * blockDim.x + threadIdx.x;
int row = blockIdx.y * blockDim.y + threadIdx.y;
```

## 2. Grid and block

A CUDA launch such as:

```cpp
kernel<<<grid, block>>>();
```

creates many GPU threads.

For convolution:

```cpp
dim3 block(16, 16);
dim3 grid((width + block.x - 1) / block.x,
          (height + block.y - 1) / block.y);
```

means each block contains `16 × 16 = 256` threads.

## 3. Device memory

Typical CUDA workflow:

```text
CPU data
   ↓
cudaMalloc()
   ↓
GPU memory
   ↓
kernel execution
   ↓
cudaMemcpy()
   ↓
CPU result
```

## 4. Synchronization

After launching a kernel:

```cpp
cudaDeviceSynchronize();
```

waits until the GPU finishes the kernel.

---

# Comparison of the Three Algorithms

| Algorithm | Parallel unit | Main data structure | Typical CUDA strategy |
|---|---|---|---|
| List Ranking | List node | `next[]`, `rank[]` | Pointer jumping |
| Sparse Matrix Computation | Matrix row | CSR | One thread per row |
| Convolution | Output pixel | Dense image + kernel | One thread per pixel |

## Complexity intuition

### List Ranking
Pointer jumping reduces the number of logical iterations to approximately:

`O(log n)`

with many nodes processed concurrently.

### CSR SpMV
The arithmetic work is proportional to the number of non-zero elements:

`O(nnz)`

rather than `O(rows × columns)`.

### 2-D Convolution
For an image with `N` pixels and a `K × K` filter:

`O(N × K²)`

operations are performed, but many output pixels can be calculated simultaneously on the GPU.

# Exercises

### Exercise 1 — List Ranking
Modify the CUDA program to support a list containing 1,024 nodes.

**Tasks**
1. Generate a sequential linked list.
2. Allocate arrays dynamically.
3. Run pointer jumping.
4. Verify that node `i` has rank `n - 1 - i`.

### Exercise 2 — Sparse Matrix
Modify the CSR SpMV program to use a `1000 × 1000` sparse matrix.

**Tasks**
1. Generate approximately 1% non-zero elements.
2. Construct `rowPtr`, `colIndex`, and `values`.
3. Perform GPU SpMV.
4. Compare GPU output against CPU output.
5. Measure execution time.

### Exercise 3 — Convolution
Replace the edge-detection filter with:

```text
[1 1 1]
[1 1 1]
[1 1 1]
```

and divide the result by 9 to implement a 3×3 averaging/blur filter.

### Exercise 4 — Optimization
Improve the convolution kernel using **shared memory** so that neighboring threads reuse image pixels instead of repeatedly reading them from global memory.

# Suggested Advanced Topics

After completing these three kernels, study:

1. CUDA memory hierarchy
2. Global vs shared vs constant memory
3. Coalesced memory access
4. Warp execution and divergence
5. Atomic operations
6. CUDA events for timing
7. Shared-memory tiled convolution
8. cuSPARSE for sparse matrix operations
9. Thrust library
10. CUDA streams and asynchronous execution
11. Pinned host memory
12. Unified Memory
13. CUDA occupancy and performance optimization